# 03 — ARIMA statistical benchmark

This notebook inspects an executed development-only run. Orders are selected by training AIC, then fixed-parameter one-step forecasts are compared with the naive benchmark on 2023–2024. No 2025 results are loaded. See [protocol and limitations](../docs/MILESTONE_4.md).


In [ ]:
from pathlib import Path
import json
import pandas as pd
import plotly.graph_objects as go
root = Path.cwd()
if not (root / "configs/nvda.json").exists():
    root = root.parent
run = root / "artifacts/arima_v1"
summary = json.loads((run / "summary.json").read_text())
predictions = pd.DataFrame(json.loads((run / "validation_predictions.json").read_text()))
predictions["target_date"] = pd.to_datetime(predictions["target_date"])
assert summary["test_evaluated"] is False
assert predictions.target_date.max() <= pd.Timestamp("2024-12-31")
pd.DataFrame(summary["candidates"])


## Selection and forecast timing

AIC is computed from training data only. All nine candidates converged; two required replacement of invalid starting parameters, as recorded in the warnings column. The selected model forecasts each target before incorporating its observed Close. Subsequent state updates use fixed coefficients and do not refit on validation targets.


In [ ]:
print("Selected order:", summary["selected_order"])
pd.DataFrame(summary["validation_metrics"]).T


In [ ]:
fig = go.Figure()
for field, label in [("actual_close", "Observed Close"), ("naive_prediction", "Naive"), ("arima_prediction", "ARIMA")]:
    fig.add_scatter(x=predictions.target_date, y=predictions[field], name=label)
fig.update_layout(title="Validation forecasts: 2023–2024", xaxis_title="Target session", yaxis_title="Split-adjusted USD", template="plotly_white")
fig.show()


In [ ]:
errors = go.Figure()
for field, label in [("naive_prediction", "Naive"), ("arima_prediction", "ARIMA")]:
    errors.add_scatter(x=predictions.target_date, y=predictions[field] - predictions.actual_close, name=label)
errors.update_layout(title="Signed validation errors (forecast minus observed)", xaxis_title="Target session", yaxis_title="Error (USD)", template="plotly_white")
errors.show()


## Interpretation

ARIMA(2,1,2) has approximately 1.82% higher validation MAE than the naive baseline. A lower training AIC does not establish better forecasting performance. Close overlap between price curves is insufficient evidence; compare aligned errors and numerical metrics. This bounded baseline is retained without searching for a favourable result. Technical-indicator and market-context LSTM experiments remain planned, with no feature implementation in this milestone.
